# Where We Left Off

In the previous lab we tracked the end-effector position in real time using forward kinematics on the MCU. In doing so we learned that the joints' state variables (in our case, the joint angles) determine the position of the end effector.

Now we want to move the end effector to a target position. To do so we have to command each motor to a given angle, but before moving any motor we need to put some safety precautions in place.

## In This Lab

| Deliverable | What you build | Torque |
| :--- | :--- | :---: |
| 2.1 | Telemetry: temperature and current of every servo | off |
| 2.2 | Safety governor: a two-state FSM that detects faults | off |
| 2.3 | Joint limits: the range of motion of each joint | off |
| 2.4 | Position control with the safety governor active | **on** |

Notice the order: we enable torque only once the robot can detect faults and knows its own limits.

# More Than Angles

Besides its position, two other quantities are important to monitor in a motor: **temperature** and **current draw**.
- Current too high → the motor is pushing against a resistance (an obstacle, a mechanical end stop, a stall) → better to stop moving.
- Temperature too high → risk of damage to the electronics and of fire → better to stop moving.

### Deliverable 2.1

**Deliverable 2.1:** write in the next cell a modified version of Deliverable 1.8 that also prints the temperature (in degrees Celsius) and the current draw (in mA) of every servo.

**Hint 1:** find the ST3215 SRAM register table at the bottom of [this web page](https://pypi.org/project/st3215/) and look for **Current temperature** and **Current current** (here *current* means *present*).

**Hint 2:** if the registers you need are close to each other, you can read one contiguous block of SRAM and simply ignore the registers in between.

**Hint 3:** the raw current value is not in mA: each unit corresponds to 6.5 mA.

**Hint 4:** the **Current current** reading will stay almost constant for now, since we are not driving the motors yet.

```text
Your code goes here.
```

### Solution

```text
// SO-101 joint reader + forward kinematics on Arduino VENTUNO Q (STM32H5 MCU)
// One SYNC READ per cycle at a fixed 30 Hz, FK computed after every read,
// printing decoupled from sensing.
// Prints the joint angles in radians (relative to ZERO_TICKS), the gripper angle,
// the end-effector position and orientation in the robot base frame,
// and the temperature and current of every servo.
//
// Waveshare Bus Servo Adapter (A) in UART mode (jumper A)
// Wiring: D1(TX)->TX, D0(RX)->RX, GND->GND
//
// Requires so101_fk.c and so101_fk.h (CasADi-generated) in the sketch folder.

#include "so101_fk.h"

// ======================= Configuration =======================

const uint32_t BAUD = 1000000;           // SO-101 default after LeRobot setup
const uint8_t  IDS[] = {1, 2, 3, 4, 5, 6};
const uint8_t  NUM_SERVOS = sizeof(IDS);
const bool     TORQUE_OFF = true;        // true = move the arm by hand
const uint16_t REPLY_TIMEOUT_US = 5000;
const uint32_t READ_PERIOD_US = 1000000UL / 30;   // read + FK at 30 Hz (33333 us)
const uint16_t PRINT_PERIOD_MS = 200;             // print at 5 Hz

const float TICKS_PER_REV = 4096.0f;
const float RAD_PER_TICK  = 2.0f * PI / TICKS_PER_REV;

// Present_Current LSB in mA (Feetech STS series)
const float MA_PER_CURRENT_UNIT = 6.5f;

// Tick value that maps to 0 rad for each servo (same calibration as the main sketch)
const int ZERO_TICKS[NUM_SERVOS] = {2093, 2091, 2040, 2026, 2056, 2046};

// ----- Forward kinematics -----
// The first NUM_FK_JOINTS entries of IDS feed q[0..4] of so101_fk, in order:
// shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll. ID 6 (gripper) is not used.
const uint8_t NUM_FK_JOINTS = 5;

// Maps servo angle to the model's joint angle: q = SIGN * servo_rad + OFFSET.
// Use -1 where the servo turns opposite to the model's joint axis, and an offset
// (rad) if your zero pose differs from the model's zero pose.
const float JOINT_SIGN[NUM_FK_JOINTS]   = { 1, 1, 1, 1, 1 };
const float JOINT_OFFSET[NUM_FK_JOINTS] = { 0, 0, 0, 0, 0 };

// ======================= STS3215 protocol =======================

const uint8_t REG_TORQUE_ENABLE       = 0x28;  // 40
const uint8_t REG_PRESENT_POSITION    = 0x38;  // 56, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_TEMPERATURE = 0x3F;  // 63, 1 byte, degrees C
const uint8_t REG_PRESENT_CURRENT     = 0x45;  // 69, 2 bytes, sign-magnitude

// One SYNC READ covers Present_Position .. Present_Current (registers 56..70)
const uint8_t READ_START = REG_PRESENT_POSITION;
const uint8_t READ_LEN   = REG_PRESENT_CURRENT + 2 - READ_START;   // 15 bytes

// Offsets of each field inside the block
const uint8_t OFS_POSITION    = REG_PRESENT_POSITION    - READ_START;
const uint8_t OFS_TEMPERATURE = REG_PRESENT_TEMPERATURE - READ_START;
const uint8_t OFS_CURRENT     = REG_PRESENT_CURRENT     - READ_START;

const uint8_t INST_WRITE     = 0x03;
const uint8_t INST_SYNC_READ = 0x82;
const uint8_t BROADCAST_ID   = 0xFE;

uint8_t lastTx[16];
uint8_t lastTxLen = 0;

// Build and send a packet; remember it so we can ignore its echo
void sendPacket(uint8_t id, uint8_t instr, const uint8_t* params, uint8_t n) {
  uint8_t len = n + 2;
  uint8_t sum = id + len + instr;
  lastTxLen = 0;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = id;
  lastTx[lastTxLen++] = len;
  lastTx[lastTxLen++] = instr;
  for (uint8_t i = 0; i < n; i++) {
    lastTx[lastTxLen++] = params[i];
    sum += params[i];
  }
  lastTx[lastTxLen++] = (uint8_t)~sum;

  while (Serial1.available()) Serial1.read();  // clear stale bytes
  Serial1.write(lastTx, lastTxLen);
  Serial1.flush();
}

// Read one byte with timeout; returns -1 on timeout
int readByte(unsigned long deadline) {
  while ((long)(deadline - micros()) > 0) {
    if (Serial1.available()) return Serial1.read();
  }
  return -1;
}

// Wait for a valid status packet from `id`, skipping our own echo.
// Fills data[] with the payload; returns payload length or -1.
int readStatus(uint8_t id, uint8_t* data, uint8_t maxData) {
  unsigned long deadline = micros() + REPLY_TIMEOUT_US;

  while ((long)(deadline - micros()) > 0) {
    // Find header 0xFF 0xFF
    int b = readByte(deadline);
    if (b != 0xFF) continue;
    b = readByte(deadline);
    if (b != 0xFF) continue;

    int rid = readByte(deadline);
    if (rid == 0xFF) rid = readByte(deadline);   // tolerate extra 0xFF
    int len = readByte(deadline);
    if (rid < 0 || len < 2 || len > maxData + 2) continue;

    uint8_t pkt[4 + READ_LEN + 2];               // header + id + len + err + data + checksum
    if (4 + len > (int)sizeof(pkt)) continue;
    uint8_t p = 0;
    pkt[p++] = 0xFF; pkt[p++] = 0xFF; pkt[p++] = rid; pkt[p++] = len;

    bool ok = true;
    for (int i = 0; i < len; i++) {
      int c = readByte(deadline);
      if (c < 0) { ok = false; break; }
      pkt[p++] = (uint8_t)c;
    }
    if (!ok) return -1;

    // Skip the echo of the packet we just sent
    if (p == lastTxLen && memcmp(pkt, lastTx, p) == 0) continue;

    // Verify checksum
    uint8_t sum = rid + len;
    for (int i = 4; i < p - 1; i++) sum += pkt[i];
    if ((uint8_t)~sum != pkt[p - 1]) continue;
    if (rid != id) continue;

    // pkt[4] = error byte, payload follows
    uint8_t n = len - 2;
    memcpy(data, &pkt[5], n);
    return n;
  }
  return -1;
}

void writeByte(uint8_t id, uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(id, INST_WRITE, p, 2);
  uint8_t dummy[4];
  readStatus(id, dummy, sizeof(dummy));  // consume the servo's ack
}

// STS 16-bit values are little-endian with bit 15 as the sign
int decodeSignMag(uint8_t lo, uint8_t hi) {
  uint16_t raw = lo | (hi << 8);
  return (raw & 0x8000) ? -(int)(raw & 0x7FFF) : (int)raw;
}

// Reads position, temperature and current of all servos with one SYNC READ.
// A servo that didn't reply gets ticks = -1 (its temperature and current are then invalid).
// Returns true if all replied.
bool readAllServos(int* ticks, int* tempC, int* currRaw) {
  uint8_t p[2 + NUM_SERVOS];
  p[0] = READ_START;
  p[1] = READ_LEN;                           // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) p[2 + i] = IDS[i];
  sendPacket(BROADCAST_ID, INST_SYNC_READ, p, sizeof(p));

  // Servos reply one after another, in the order of the ID list
  bool ok = true;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    uint8_t data[READ_LEN];
    int n = readStatus(IDS[i], data, sizeof(data));
    if (n != READ_LEN) {
      ticks[i] = -1; tempC[i] = -1; currRaw[i] = 0;
      ok = false;
      continue;
    }
    ticks[i]   = decodeSignMag(data[OFS_POSITION], data[OFS_POSITION + 1]);
    tempC[i]   = data[OFS_TEMPERATURE];
    currRaw[i] = decodeSignMag(data[OFS_CURRENT], data[OFS_CURRENT + 1]);
  }
  return ok;
}

// Converts a servo's raw ticks to radians relative to its zero point
float ticksToRad(uint8_t i, int ticks) {
  return (ticks - ZERO_TICKS[i]) * RAD_PER_TICK;
}

// ======================= Forward kinematics =======================

// Wraps the CasADi function so101_fk:(q[5]) -> (p[3], R[3x3]).
// p is in meters, in the robot base frame.
// R is stored column-major (CasADi convention): R(row, col) = R[row + 3*col].
bool computeFK(const float* q, float* p, float* R) {
  const casadi_real* arg[so101_fk_SZ_ARG] = { q };
  casadi_real*       res[so101_fk_SZ_RES] = { p, R };
  // SZ_IW and SZ_W are 0, so no work buffers are needed
  return so101_fk(arg, res, NULL, NULL, 0) == 0;
}

inline float Rat(const float* R, int row, int col) { return R[row + 3 * col]; }

// ZYX Euler angles (roll about x, pitch about y, yaw about z), in radians
void rotToRPY(const float* R, float& roll, float& pitch, float& yaw) {
  yaw   = atan2f(Rat(R, 1, 0), Rat(R, 0, 0));
  pitch = atan2f(-Rat(R, 2, 0), sqrtf(Rat(R, 2, 1) * Rat(R, 2, 1) + Rat(R, 2, 2) * Rat(R, 2, 2)));
  roll  = atan2f(Rat(R, 2, 1), Rat(R, 2, 2));
}

// Prints a value with a leading space for positives so columns line up
void printAligned(float v, uint8_t decimals) {
  if (v >= 0) Serial.print(' ');
  Serial.print(v, decimals);
}

// Prints an integer right-aligned in a field of `width` characters
void printPadded(long v, uint8_t width) {
  char buf[16];
  snprintf(buf, sizeof(buf), "%*ld", width, v);
  Serial.print(buf);
}

// ======================= Main =======================

void setup() {
  Serial.begin(115200);
  Serial1.begin(BAUD);
  delay(200);

  // Sanity check that the generated function matches what this sketch expects
  const casadi_int* sp = so101_fk_sparsity_in(0);
  if (so101_fk_n_in() != 1 || so101_fk_n_out() != 2 || sp[0] != NUM_FK_JOINTS) {
    Serial.println("ERROR: so101_fk signature does not match NUM_FK_JOINTS");
  }

  if (TORQUE_OFF) {
    for (uint8_t i = 0; i < NUM_SERVOS; i++) {
      writeByte(IDS[i], REG_TORQUE_ENABLE, 0);
    }
  }
}

void loop() {
  static unsigned long lastPrint = 0;
  static unsigned long nextRead  = micros();

  // Kept between iterations: most loop() calls don't read, but may print
  static int ticks[NUM_SERVOS]   = {-1, -1, -1, -1, -1, -1};
  static int tempC[NUM_SERVOS]   = {-1, -1, -1, -1, -1, -1};
  static int currRaw[NUM_SERVOS] = {0};

  // Last good FK result is kept so a dropped reply doesn't blank the output
  static float    q[NUM_FK_JOINTS] = {0};
  static float    p[3] = {0, 0, 0};
  static float    R[9] = {1, 0, 0,  0, 1, 0,  0, 0, 1};
  static bool     haveFK = false;    // at least one FK result available
  static bool     fresh  = false;    // FK succeeded on the most recent read
  static uint32_t fkMicros = 0;      // time taken by the last FK evaluation

  // ---- Read + FK at a fixed 30 Hz ----
  // The deadline advances by exactly one period, so the rate doesn't drift
  // even if a cycle starts a little late.
  if ((long)(micros() - nextRead) >= 0) {
    nextRead += READ_PERIOD_US;
    // If we fell more than a full period behind, resynchronize instead of
    // firing a burst of back-to-back reads to catch up
    if ((long)(micros() - nextRead) >= 0) nextRead = micros() + READ_PERIOD_US;

    readAllServos(ticks, tempC, currRaw);

    // Joint angles -> FK (only when all arm joints replied)
    bool armOk = true;
    for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
      if (ticks[i] == -1) { armOk = false; break; }
    }

    if (armOk) {
      for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
        q[i] = JOINT_SIGN[i] * ticksToRad(i, ticks[i]) + JOINT_OFFSET[i];
      }
      uint32_t t0 = micros();
      fresh = computeFK(q, p, R);
      fkMicros = micros() - t0;
      haveFK = haveFK || fresh;
    } else {
      fresh = false;
    }
  }

  // ---- Printing at a fixed rate ----
  if (millis() - lastPrint >= PRINT_PERIOD_MS) {
    lastPrint = millis();

    // Line 1: joints, gripper, FK
    Serial.print("q[rad] ");
    for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
      printAligned(q[i], 3);
      Serial.print(' ');
    }

    Serial.print("| grip ");
    if (ticks[NUM_SERVOS - 1] == -1) Serial.print("  N/A ");
    else printAligned(ticksToRad(NUM_SERVOS - 1, ticks[NUM_SERVOS - 1]), 3);

    if (haveFK) {
      float roll, pitch, yaw;
      rotToRPY(R, roll, pitch, yaw);

      Serial.print(" | xyz[mm] ");
      for (uint8_t k = 0; k < 3; k++) {
        printAligned(p[k] * 1000.0f, 1);
        Serial.print(' ');
      }
      Serial.print("| rpy[deg] ");
      printAligned(roll  * RAD_TO_DEG, 1); Serial.print(' ');
      printAligned(pitch * RAD_TO_DEG, 1); Serial.print(' ');
      printAligned(yaw   * RAD_TO_DEG, 1);
      Serial.print(" | fk ");
      Serial.print(fkMicros);
      Serial.print("us");
      if (!fresh) Serial.print(" (stale)");
    } else {
      Serial.print(" | FK: waiting for all joints");
    }
    Serial.println();

    // Line 2: temperature and current per servo (order of IDS)
    Serial.print("T[C]   ");
    for (uint8_t i = 0; i < NUM_SERVOS; i++) {
      if (ticks[i] == -1) Serial.print(" N/A");
      else printPadded(tempC[i], 4);
    }
    Serial.print(" | I[mA] ");
    long totalmA = 0;
    for (uint8_t i = 0; i < NUM_SERVOS; i++) {
      if (ticks[i] == -1) { Serial.print("   N/A"); continue; }
      long mA = lroundf(currRaw[i] * MA_PER_CURRENT_UNIT);
      totalmA += labs(mA);
      printPadded(mA, 6);
    }
    Serial.print(" | total ");
    Serial.print(totalmA);
    Serial.println("mA");
  }
}
```

# The Safety Governor

We want to keep every motor within its working conditions, so we build a Finite State Machine (FSM) that halts the robot whenever a motor leaves its safe operating range for temperature or current, or stops responding.

According to the [ST3215 product page](https://www.dfrobot.com/product-2962.html), the working temperature should not exceed 60 °C, and the maximum current is 2700 mA. Whenever one of these limits is exceeded, or at least one servo does not respond for more than 5 consecutive readings, the FSM must switch to the halt state; otherwise it stays in normal operation.

> **Note on the current limit:** 2700 mA is close to the servo's stall current, i.e. the current it draws when it pushes as hard as it can without moving. A governor using this threshold will therefore only trip when a motor is (almost) fully stalled. We keep this value for now and come back to it in Deliverable 2.4.

> **Design choice: HALT is latching.** Once the robot is in HALT, it stays there until the MCU is restarted, even if the fault disappears (e.g. a servo cools down or a cable is plugged back in). A fault that clears by itself does not mean its cause has been fixed: a person should inspect the robot before it moves again.

### Deliverable 2.2

**Deliverable 2.2:** write in the next cell a modified version of Deliverable 2.1 that implements a Finite State Machine (FSM) with two states:
- a **normal operation state**, in which the robot operates as planned
- a **halt state**, in which the robot stops its operations

There is only one transition in this FSM, from the normal state to the halt state. It is taken whenever at least one of the following is true:
- at least one servo's temperature is above 60 °C
- at least one servo draws more than 2700 mA (in absolute value)
- at least one servo has not responded for more than 5 consecutive readings

For now, neither state does anything.

The program shall print the current state name, at 5 Hz and, when the transition happens, the reason(s) for it.

**Hint:** the current is a signed value (sign-magnitude encoding), so compare its absolute value with the limit.

**Test:** with the program running, gently unplug one servo cable and check that the state changes to `HALT` and that the reason is printed. The servos are daisy-chained, so unplugging a cable also disconnects every servo further down the chain: expect one *not responding* reason for each disconnected servo. Torque is still off in this deliverable, so the arm will not move.

```text
Your code goes here.
```

### Solution

```text
// SO-101 servo monitor with a two-state FSM and forward kinematics
// on Arduino VENTUNO Q (STM32H5 MCU)
//
// States:
//   NORMAL - the robot operates as planned (placeholder for now)
//   HALT   - the robot stops its operations (placeholder for now, terminal)
//
// Single transition NORMAL -> HALT, taken when any of these is true:
//   - a servo temperature is above MAX_TEMP_C
//   - a servo current (absolute value) is above MAX_CURRENT_MA
//   - a servo has missed more than MAX_MISSED_READS consecutive readings
//
// Output at 5 Hz, one line per print:
//   state | joint angles [rad] | gripper | end-effector position and orientation
//         | temperature and current of every servo
// plus the reason(s) when the transition to HALT happens.
//
// Waveshare Bus Servo Adapter (A) in UART mode (jumper A)
// Wiring: D1(TX)->TX, D0(RX)->RX, GND->GND
//
// Requires so101_fk.c and so101_fk.h (CasADi-generated) in the sketch folder.

#include "so101_fk.h"

// ======================= Configuration =======================

const uint32_t BAUD = 1000000;           // SO-101 default after LeRobot setup
const uint8_t  IDS[] = {1, 2, 3, 4, 5, 6};
const uint8_t  NUM_SERVOS = sizeof(IDS);
const bool     TORQUE_OFF = true;        // true = move the arm by hand
const uint16_t REPLY_TIMEOUT_US = 5000;
const uint32_t READ_PERIOD_US = 1000000UL / 30;   // sensing + FK at 30 Hz (33333 us)
const uint16_t PRINT_PERIOD_MS = 200;             // print at 5 Hz

const float TICKS_PER_REV = 4096.0f;
const float RAD_PER_TICK  = 2.0f * PI / TICKS_PER_REV;

// Present_Current LSB in mA (Feetech STS series)
const float MA_PER_CURRENT_UNIT = 6.5f;

// Tick value that maps to 0 rad for each servo (same calibration as the main sketch)
const int ZERO_TICKS[NUM_SERVOS] = {2093, 2091, 2040, 2026, 2056, 2046};

// ----- Halt conditions -----
const int     MAX_TEMP_C       = 60;     // halt if any servo is hotter than this
const long    MAX_CURRENT_MA   = 2700;   // halt if any servo draws more than this
const uint8_t MAX_MISSED_READS = 5;      // halt if any servo misses more than this many reads in a row

// ----- Forward kinematics -----
// The first NUM_FK_JOINTS entries of IDS feed q[0..4] of so101_fk, in order:
// shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll. ID 6 (gripper) is not used.
const uint8_t NUM_FK_JOINTS = 5;

// Maps servo angle to the model's joint angle: q = SIGN * servo_rad + OFFSET.
// Use -1 where the servo turns opposite to the model's joint axis, and an offset
// (rad) if your zero pose differs from the model's zero pose.
const float JOINT_SIGN[NUM_FK_JOINTS]   = { 1, 1, 1, 1, 1 };
const float JOINT_OFFSET[NUM_FK_JOINTS] = { 0, 0, 0, 0, 0 };

// ======================= FSM types =======================
// Defined before any function: the Arduino builder inserts auto-generated
// function prototypes above the first function, and they need these types.

enum State : uint8_t { STATE_NORMAL, STATE_HALT };

enum FaultType : uint8_t { FAULT_OVER_TEMP, FAULT_OVER_CURRENT, FAULT_NO_REPLY };

struct Fault {
  FaultType type;
  uint8_t   id;      // servo ID
  long      value;   // degC, mA, or number of consecutive missed reads
};

// ======================= STS3215 protocol =======================

const uint8_t REG_TORQUE_ENABLE       = 0x28;  // 40
const uint8_t REG_PRESENT_POSITION    = 0x38;  // 56, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_TEMPERATURE = 0x3F;  // 63, 1 byte, degrees C
const uint8_t REG_PRESENT_CURRENT     = 0x45;  // 69, 2 bytes, sign-magnitude

// One SYNC READ covers Present_Position .. Present_Current (registers 56..70)
const uint8_t READ_START = REG_PRESENT_POSITION;
const uint8_t READ_LEN   = REG_PRESENT_CURRENT + 2 - READ_START;   // 15 bytes

// Offsets of each field inside the block
const uint8_t OFS_POSITION    = REG_PRESENT_POSITION    - READ_START;
const uint8_t OFS_TEMPERATURE = REG_PRESENT_TEMPERATURE - READ_START;
const uint8_t OFS_CURRENT     = REG_PRESENT_CURRENT     - READ_START;

const uint8_t INST_WRITE     = 0x03;
const uint8_t INST_SYNC_READ = 0x82;
const uint8_t BROADCAST_ID   = 0xFE;

uint8_t lastTx[16];
uint8_t lastTxLen = 0;

// Build and send a packet; remember it so we can ignore its echo
void sendPacket(uint8_t id, uint8_t instr, const uint8_t* params, uint8_t n) {
  uint8_t len = n + 2;
  uint8_t sum = id + len + instr;
  lastTxLen = 0;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = id;
  lastTx[lastTxLen++] = len;
  lastTx[lastTxLen++] = instr;
  for (uint8_t i = 0; i < n; i++) {
    lastTx[lastTxLen++] = params[i];
    sum += params[i];
  }
  lastTx[lastTxLen++] = (uint8_t)~sum;

  while (Serial1.available()) Serial1.read();  // clear stale bytes
  Serial1.write(lastTx, lastTxLen);
  Serial1.flush();
}

// Read one byte with timeout; returns -1 on timeout
int readByte(unsigned long deadline) {
  while ((long)(deadline - micros()) > 0) {
    if (Serial1.available()) return Serial1.read();
  }
  return -1;
}

// Wait for a valid status packet from `id`, skipping our own echo.
// Fills data[] with the payload; returns payload length or -1.
int readStatus(uint8_t id, uint8_t* data, uint8_t maxData) {
  unsigned long deadline = micros() + REPLY_TIMEOUT_US;

  while ((long)(deadline - micros()) > 0) {
    // Find header 0xFF 0xFF
    int b = readByte(deadline);
    if (b != 0xFF) continue;
    b = readByte(deadline);
    if (b != 0xFF) continue;

    int rid = readByte(deadline);
    if (rid == 0xFF) rid = readByte(deadline);   // tolerate extra 0xFF
    int len = readByte(deadline);
    if (rid < 0 || len < 2 || len > maxData + 2) continue;

    uint8_t pkt[4 + READ_LEN + 2];               // header + id + len + err + data + checksum
    if (4 + len > (int)sizeof(pkt)) continue;
    uint8_t p = 0;
    pkt[p++] = 0xFF; pkt[p++] = 0xFF; pkt[p++] = rid; pkt[p++] = len;

    bool ok = true;
    for (int i = 0; i < len; i++) {
      int c = readByte(deadline);
      if (c < 0) { ok = false; break; }
      pkt[p++] = (uint8_t)c;
    }
    if (!ok) return -1;

    // Skip the echo of the packet we just sent
    if (p == lastTxLen && memcmp(pkt, lastTx, p) == 0) continue;

    // Verify checksum
    uint8_t sum = rid + len;
    for (int i = 4; i < p - 1; i++) sum += pkt[i];
    if ((uint8_t)~sum != pkt[p - 1]) continue;
    if (rid != id) continue;

    // pkt[4] = error byte, payload follows
    uint8_t n = len - 2;
    memcpy(data, &pkt[5], n);
    return n;
  }
  return -1;
}

void writeByte(uint8_t id, uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(id, INST_WRITE, p, 2);
  uint8_t dummy[4];
  readStatus(id, dummy, sizeof(dummy));  // consume the servo's ack
}

// STS 16-bit values are little-endian with bit 15 as the sign
int decodeSignMag(uint8_t lo, uint8_t hi) {
  uint16_t raw = lo | (hi << 8);
  return (raw & 0x8000) ? -(int)(raw & 0x7FFF) : (int)raw;
}

// Reads position, temperature and current of all servos with one SYNC READ.
// replied[i] tells whether servo i answered; its other fields are valid only if so.
// Returns true if all replied.
bool readAllServos(int* ticks, int* tempC, int* currRaw, bool* replied) {
  uint8_t p[2 + NUM_SERVOS];
  p[0] = READ_START;
  p[1] = READ_LEN;                           // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) p[2 + i] = IDS[i];
  sendPacket(BROADCAST_ID, INST_SYNC_READ, p, sizeof(p));

  // Servos reply one after another, in the order of the ID list
  bool ok = true;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    uint8_t data[READ_LEN];
    int n = readStatus(IDS[i], data, sizeof(data));
    replied[i] = (n == READ_LEN);
    if (!replied[i]) {
      ok = false;
      continue;
    }
    ticks[i]   = decodeSignMag(data[OFS_POSITION], data[OFS_POSITION + 1]);
    tempC[i]   = data[OFS_TEMPERATURE];
    currRaw[i] = decodeSignMag(data[OFS_CURRENT], data[OFS_CURRENT + 1]);
  }
  return ok;
}

// Converts a servo's raw ticks to radians relative to its zero point
float ticksToRad(uint8_t i, int ticks) {
  return (ticks - ZERO_TICKS[i]) * RAD_PER_TICK;
}

// ======================= Forward kinematics =======================

// Wraps the CasADi function so101_fk:(q[5]) -> (p[3], R[3x3]).
// p is in meters, in the robot base frame.
// R is stored column-major (CasADi convention): R(row, col) = R[row + 3*col].
bool computeFK(const float* q, float* p, float* R) {
  const casadi_real* arg[so101_fk_SZ_ARG] = { q };
  casadi_real*       res[so101_fk_SZ_RES] = { p, R };
  // SZ_IW and SZ_W are 0, so no work buffers are needed
  return so101_fk(arg, res, NULL, NULL, 0) == 0;
}

inline float Rat(const float* R, int row, int col) { return R[row + 3 * col]; }

// ZYX Euler angles (roll about x, pitch about y, yaw about z), in radians
void rotToRPY(const float* R, float& roll, float& pitch, float& yaw) {
  yaw   = atan2f(Rat(R, 1, 0), Rat(R, 0, 0));
  pitch = atan2f(-Rat(R, 2, 0), sqrtf(Rat(R, 2, 1) * Rat(R, 2, 1) + Rat(R, 2, 2) * Rat(R, 2, 2)));
  roll  = atan2f(Rat(R, 2, 1), Rat(R, 2, 2));
}

// ======================= Finite State Machine =======================

const char* stateName(State s) {
  switch (s) {
    case STATE_NORMAL: return "NORMAL";
    case STATE_HALT:   return "HALT";
  }
  return "UNKNOWN";
}

// A servo can trip at most two conditions at once (temp + current, or no reply)
const uint8_t MAX_FAULTS = 2 * NUM_SERVOS;
Fault   faults[MAX_FAULTS];
uint8_t numFaults = 0;

State state = STATE_NORMAL;

// Latest sensor data (valid for servo i only if replied[i] is true)
int  ticks[NUM_SERVOS];
int  tempC[NUM_SERVOS];
int  currRaw[NUM_SERVOS];
bool replied[NUM_SERVOS];

// Consecutive missed readings per servo
uint8_t missedReads[NUM_SERVOS] = {0};

// Latest kinematics. The last good FK result is kept so a dropped reply
// doesn't blank the output.
float    q[NUM_FK_JOINTS] = {0};
float    eePos[3] = {0, 0, 0};                     // end-effector position [m]
float    eeRot[9] = {1, 0, 0,  0, 1, 0,  0, 0, 1}; // end-effector rotation, column-major
bool     haveFK   = false;   // at least one FK result available
bool     fkFresh  = false;   // FK succeeded on the most recent read
uint32_t fkMicros = 0;       // time taken by the last FK evaluation

// Converts the latest reading to joint angles and runs FK.
// FK runs only when all arm joints replied; otherwise the last result is kept.
void updateKinematics() {
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    if (!replied[i]) { fkFresh = false; return; }
  }
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    q[i] = JOINT_SIGN[i] * ticksToRad(i, ticks[i]) + JOINT_OFFSET[i];
  }
  uint32_t t0 = micros();
  fkFresh  = computeFK(q, eePos, eeRot);
  fkMicros = micros() - t0;
  haveFK   = haveFK || fkFresh;
}

void addFault(FaultType type, uint8_t id, long value) {
  if (numFaults < MAX_FAULTS) faults[numFaults++] = { type, id, value };
}

// Guard of the NORMAL -> HALT transition, evaluated on each new reading.
// Records every condition that is currently true; returns true if any.
bool checkHaltConditions() {
  numFaults = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) {
      if (missedReads[i] < 255) missedReads[i]++;
      if (missedReads[i] > MAX_MISSED_READS) {
        addFault(FAULT_NO_REPLY, IDS[i], missedReads[i]);
      }
      continue;   // temperature and current are not valid for this servo
    }
    missedReads[i] = 0;

    if (tempC[i] > MAX_TEMP_C) {
      addFault(FAULT_OVER_TEMP, IDS[i], tempC[i]);
    }

    long mA = labs(lroundf(currRaw[i] * MA_PER_CURRENT_UNIT));
    if (mA > MAX_CURRENT_MA) {
      addFault(FAULT_OVER_CURRENT, IDS[i], mA);
    }
  }
  return numFaults > 0;
}

void printFault(const Fault& f) {
  Serial.print("  reason: servo ");
  Serial.print(f.id);
  switch (f.type) {
    case FAULT_OVER_TEMP:
      Serial.print(" temperature ");
      Serial.print(f.value);
      Serial.print(" C > ");
      Serial.print(MAX_TEMP_C);
      Serial.println(" C");
      break;
    case FAULT_OVER_CURRENT:
      Serial.print(" current ");
      Serial.print(f.value);
      Serial.print(" mA > ");
      Serial.print(MAX_CURRENT_MA);
      Serial.println(" mA");
      break;
    case FAULT_NO_REPLY:
      Serial.print(" not responding for ");
      Serial.print(f.value);
      Serial.print(" consecutive readings > ");
      Serial.println(MAX_MISSED_READS);
      break;
  }
}

void transitionTo(State next) {
  Serial.print("TRANSITION ");
  Serial.print(stateName(state));
  Serial.print(" -> ");
  Serial.println(stateName(next));
  if (next == STATE_HALT) {
    for (uint8_t k = 0; k < numFaults; k++) printFault(faults[k]);
  }
  state = next;
}

// ----- State behaviours (placeholders) -----

void runNormalState() {
  // Normal operation goes here (e.g. control). Nothing for now.
}

void runHaltState() {
  // Halt behaviour goes here (e.g. disable torque). Nothing for now.
}

// ======================= Printing =======================

// Prints a value with a leading space for positives so columns line up
void printAligned(float v, uint8_t decimals) {
  if (v >= 0) Serial.print(' ');
  Serial.print(v, decimals);
}

// Prints an integer right-aligned in a field of `width` characters
void printPadded(long v, uint8_t width) {
  char buf[16];
  snprintf(buf, sizeof(buf), "%*ld", width, v);
  Serial.print(buf);
}

// Prints one status line: state, joints/FK, temperature/current
void printStatus() {
  // State, padded so the columns stay put when it changes
  char st[16];
  snprintf(st, sizeof(st), "%-6s", stateName(state));
  Serial.print("State: ");
  Serial.print(st);

  // Joints, gripper, FK
  Serial.print(" | q[rad] ");
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    printAligned(q[i], 3);
    Serial.print(' ');
  }

  const uint8_t g = NUM_SERVOS - 1;   // gripper index
  Serial.print("| grip ");
  if (!replied[g]) Serial.print("  N/A ");
  else printAligned(ticksToRad(g, ticks[g]), 3);

  if (haveFK) {
    float roll, pitch, yaw;
    rotToRPY(eeRot, roll, pitch, yaw);

    Serial.print(" | xyz[mm] ");
    for (uint8_t k = 0; k < 3; k++) {
      printAligned(eePos[k] * 1000.0f, 1);
      Serial.print(' ');
    }
    Serial.print("| rpy[deg] ");
    printAligned(roll  * RAD_TO_DEG, 1); Serial.print(' ');
    printAligned(pitch * RAD_TO_DEG, 1); Serial.print(' ');
    printAligned(yaw   * RAD_TO_DEG, 1);
    Serial.print(" | fk ");
    Serial.print(fkMicros);
    Serial.print("us");
    if (!fkFresh) Serial.print(" (stale)");
  } else {
    Serial.print(" | FK: waiting for all joints");
  }

  // Temperature and current per servo (order of IDS)
  Serial.print(" | T[C] ");
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) Serial.print(" N/A");
    else printPadded(tempC[i], 4);
  }
  Serial.print(" | I[mA] ");
  long totalmA = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) { Serial.print("   N/A"); continue; }
    long mA = lroundf(currRaw[i] * MA_PER_CURRENT_UNIT);
    totalmA += labs(mA);
    printPadded(mA, 6);
  }
  Serial.print(" | total ");
  Serial.print(totalmA);
  Serial.println("mA");
}

// ======================= Main =======================

void setup() {
  Serial.begin(115200);
  Serial1.begin(BAUD);
  delay(200);

  // Sanity check that the generated function matches what this sketch expects
  const casadi_int* sp = so101_fk_sparsity_in(0);
  if (so101_fk_n_in() != 1 || so101_fk_n_out() != 2 || sp[0] != NUM_FK_JOINTS) {
    Serial.println("ERROR: so101_fk signature does not match NUM_FK_JOINTS");
  }

  if (TORQUE_OFF) {
    for (uint8_t i = 0; i < NUM_SERVOS; i++) {
      writeByte(IDS[i], REG_TORQUE_ENABLE, 0);
    }
  }
}

void loop() {
  static unsigned long lastPrint = 0;
  static unsigned long nextRead  = micros();

  switch (state) {
    case STATE_NORMAL: {
      // Sensing at a fixed 30 Hz feeds the NORMAL -> HALT guard.
      // The deadline advances by exactly one period, so the rate doesn't drift.
      if ((long)(micros() - nextRead) >= 0) {
        nextRead += READ_PERIOD_US;
        // If we fell more than a full period behind, resynchronize
        if ((long)(micros() - nextRead) >= 0) nextRead = micros() + READ_PERIOD_US;

        readAllServos(ticks, tempC, currRaw, replied);
        updateKinematics();

        if (checkHaltConditions()) {
          transitionTo(STATE_HALT);
          break;
        }
      }
      runNormalState();
      break;
    }

    case STATE_HALT:
      runHaltState();
      break;
  }

  // ---- Print the status line at 5 Hz ----
  if (millis() - lastPrint >= PRINT_PERIOD_MS) {
    lastPrint = millis();
    printStatus();
  }
}
```

# Redundancy

The same limits have also been set directly in the motors' firmware by writing the ST3215 EEPROM registers. We don't do this in the lab to avoid wasting EEPROM write cycles. More information can be found in [this ST3215 library](https://pypi.org/project/st3215/).

The two layers complement each other: the firmware protects each servo on its own, even if the MCU crashes, while our governor sees the whole robot and can stop every servo as soon as any one of them is in trouble.

# Moving the Motors for the First Time

## Startup

To move a motor we must enable its torque by writing 1 to register 0x28 (**Torque switch**). As soon as torque is enabled, the servo drives towards whatever position is stored in its goal register. So before enabling torque, we set the desired position equal to the present position, to avoid sudden and unexpected movements. The desired position, in ticks in the range [0, 4095], is written to register 0x2A (**Target location**).

## Reaching the Desired Position

Once torque is enabled, the motor moves to the target location using its internal PID controller, whose parameters are stored in the servo's EEPROM. Suitable PID parameters have already been set by the instructors.

Keep in mind that a new target is reached as quickly as the servo's controller allows: a large change in the target means a fast, large movement. This is why the test sequence of Deliverable 2.4 uses small steps. Ways to make motion smoother are discussed in the optional extension at the end of this notebook.

## Limiting the Range of Motion According to the Robot's Geometry

We do not want any motor to try to reach an angle outside the range of motion of the robot. Therefore we need a function that clamps the desired angle to the range of motion of each motor. To build it, we first have to measure the minimum and maximum position of each joint.

### Deliverable 2.3

**Deliverable 2.3:** fill in the following table with the minimum and maximum position, in ticks, that each joint can reach.

**Hint:** use script A from Lab 1 to print each joint position in ticks (torque off), then move each joint by hand to its upper and lower limit. Move gently: the limit is where the joint stops moving freely, so don't force it any further. Also note which physical direction corresponds to the maximum and which to the minimum (for example, gripper open or closed).

| **Motor**        | ID1   | ID2   | ID3   | ID4   | ID5   | ID6   |
| :---             | :---: | :---: | :---: | :---: | :---: | :---: |
| **Max Position** |       |       |       |       |       |       |
| **Min Position** |       |       |       |       |       |       |

### Solution

| **Motor**        | ID1   | ID2   | ID3   | ID4   | ID5   | ID6   |
| :---             | :---: | :---: | :---: | :---: | :---: | :---: |
| **Max Position** | 3252  | 3244  | 3109  | 3217  | 3936  | 3500  |
| **Min Position** | 803   | 815   | 910   | 882   | 117   | 2043  |

> **Think about it:** these values are the mechanical end stops of each joint. If a joint is commanded exactly to its end stop and held there, the servo may keep pushing against the stop, drawing a high current and heating up. In a real deployment you would shrink each range by a small safety margin. The reference solution of Deliverable 2.4 uses the measured values directly, so be careful when commanding angles close to the limits. This is especially true for the gripper: its zero position (2046 ticks) is only a few ticks away from its closed limit (2043 ticks).

## Redundancy, Again

Similar position limits have also been set directly in the motors' firmware by writing the ST3215 EEPROM registers. We don't do this in the lab to avoid wasting EEPROM write cycles. More information can be found in [this ST3215 library](https://pypi.org/project/st3215/).

## Finally Moving the Robot

Now that we know the limits of each motor, we can safely write the **Target location** register of each motor with torque enabled. Remember that we reason in angles, so each desired angle has to be converted into a position in ticks; you already know how to do this from Deliverable 1.7.

### Deliverable 2.4

**Deliverable 2.4:** write in the next cell a modified version of Deliverable 2.2 that:
- in the **normal operation state**, moves each motor to a target angle typed in the terminal, clamped to the limits measured in Deliverable 2.3; if something else is written, go to the halt state.
- in the **halt state**, disables the torque of every motor.

During setup, before enabling the torque, make sure that each motor's target position matches its present position.

Target angles are relative to the zero positions of Deliverable 1.7 and are typed in **radians**, as six values separated by spaces (one per servo, in ID order).

> **Heads-up:** disabling the torque is safe for the motors, but not necessarily for the arm. Without torque, gravity takes over and the arm collapses. Keep a hand near the arm whenever the program is running.

**Test sequence:**

1. Put the robot in the starting pose shown below. This pose should correspond to the zero angles of Deliverable 1.7.

```text
       ╭─────┬────────────────────┬──────╮ ◉╲═════╗   ← moveable claw
       │     │      forearm       │  ▤▤  │╤══╲════╝   ← fixed claw
       ╰┬───┬┴────────────────────┴──────┴┴═══════╝
        │   │
        │   │
        │   │
        │   │   upper arm
        │   │
        │   │
        │   │
   ╭────┴───┴────╮
   │   base ◉    │
   ╰──┬───────┬──╯
  ╭───┴───────┴───╮
  │▓▓▓ C-clamp ▓▓▓│
  ╰───────────────╯
 ═══════════════════════════════════════════════════
                table edge
```

2. Launch the MCU program.
3. Hold the arm and try to move it by hand to trigger the safety governor. **Be ready to catch the arm:** when the governor halts, the torque is removed and the arm falls. If you can't trip the governor by pushing, that is a finding in itself (see the note on the current limit in the Safety Governor section). In that case, trigger it instead by unplugging the bus cable from the adapter while supporting the arm.
4. Stop the MCU program.
5. Put the robot back in the starting pose.
6. Launch the MCU program. Check the start angles printed at setup: they should all be close to 0. If any of them is far from 0, **do not** send the next command, because that joint would jump to zero at full speed. Fix the pose (or the calibration) first.
7. Send the target angles `0 0 0 0 0 0`.
8. Send a small angle change to the gripper: `0 0 0 0 0 0.17` (about 10°).
9. Send a bigger angle change to the gripper: `0 0 0 0 0 1.57` (about 90°).
10. Return the gripper to its initial angle: `0 0 0 0 0 0`.
11. Move another joint by a small angle, between -0.17 and +0.17 rad (about ±10°), e.g. `0 0 0.1 0 0 0`.
12. Return to the initial angles: `0 0 0 0 0 0`.
13. Repeat steps 11 and 12 for the other joints.
14. Put the safety controller in the halt state by typing "halt"

**Note:** the gripper's zero is (almost) its closed position, so negative gripper angles are outside its range and will be clamped.

```text
Your code goes here.
```

### Solution

```text
// SO-101 position control with a two-state FSM on Arduino VENTUNO Q (STM32H5 MCU)
//
// States:
//   NORMAL - every servo tracks a target angle typed in the terminal,
//            clamped to the joint limits measured in Deliverable 2.3
//   HALT   - torque is disabled on every servo (terminal state)
//
// Transition NORMAL -> HALT when any of these is true:
//   - a servo temperature is above MAX_TEMP_C
//   - a servo current (absolute value) is above MAX_CURRENT_MA
//   - a servo has missed more than MAX_MISSED_READS consecutive readings
//   - setup could not align a servo's goal with its present position
//   - the terminal receives anything other than exactly 6 angles
//
// Terminal command (one per line, angles relative to ZERO_TICKS):
//   <a1> <a2> ... <a6>      e.g. "0 0 0 0 0 0.3"   -> move all servos (order of IDS)
// Angles are in radians. Any other non-empty line (e.g. "halt", "3 0.5",
// "abc", 5 or 7 values) sends the robot to HALT.
//
// Output at 5 Hz, one line per print:
//   state | joint angles [rad] | gripper | end-effector position and orientation
//         | temperature and current of every servo
// plus targets, transitions and their reasons when they happen.
//
// Waveshare Bus Servo Adapter (A) in UART mode (jumper A)
// Wiring: D1(TX)->TX, D0(RX)->RX, GND->GND
//
// Requires so101_fk.c and so101_fk.h (CasADi-generated) in the sketch folder.

#include "so101_fk.h"

// ======================= Configuration =======================

const uint32_t BAUD = 1000000;           // SO-101 default after LeRobot setup
const uint8_t  IDS[] = {1, 2, 3, 4, 5, 6};
const uint8_t  NUM_SERVOS = sizeof(IDS);
const uint16_t REPLY_TIMEOUT_US = 5000;
const uint32_t CYCLE_PERIOD_US = 1000000UL / 30;  // write targets + read + FK + guard at 30 Hz
const uint16_t PRINT_PERIOD_MS = 200;             // print status line at 5 Hz
const uint16_t HALT_REPEAT_MS  = 100;             // HALT re-sends torque off at 10 Hz
const uint8_t  MAX_SETUP_ATTEMPTS = 10;           // tries to align goals with present positions

const float TICKS_PER_REV = 4096.0f;
const float RAD_PER_TICK  = 2.0f * PI / TICKS_PER_REV;

// Present_Current LSB in mA (Feetech STS series)
const float MA_PER_CURRENT_UNIT = 6.5f;

// Angles typed in the terminal are in radians, relative to ZERO_TICKS

// Tick value that maps to angle 0 for each servo (order of IDS)
const int ZERO_TICKS[NUM_SERVOS] = {2093, 2091, 2040, 2026, 2056, 2046};

// Joint limits from Deliverable 2.3, in ticks (order of IDS)
const int MIN_TICKS[NUM_SERVOS] = { 803,  815,  910,  882,  117, 2043};
const int MAX_TICKS[NUM_SERVOS] = {3252, 3244, 3109, 3217, 3936, 3500};

// ----- Halt conditions -----
const int     MAX_TEMP_C       = 60;     // halt if any servo is hotter than this
const long    MAX_CURRENT_MA   = 2700;   // halt if any servo draws more than this
const uint8_t MAX_MISSED_READS = 5;      // halt if any servo misses more than this many reads in a row

const uint8_t INPUT_LINE_LEN = 80;       // max length of a terminal command

// ----- Forward kinematics (for the status line) -----
// The first NUM_FK_JOINTS entries of IDS feed q[0..4] of so101_fk, in order:
// shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll. ID 6 (gripper) is not used.
const uint8_t NUM_FK_JOINTS = 5;

// Maps servo angle to the model's joint angle: q = SIGN * servo_rad + OFFSET.
// Use -1 where the servo turns opposite to the model's joint axis, and an offset
// (rad) if your zero pose differs from the model's zero pose.
const float JOINT_SIGN[NUM_FK_JOINTS]   = { 1, 1, 1, 1, 1 };
const float JOINT_OFFSET[NUM_FK_JOINTS] = { 0, 0, 0, 0, 0 };

// ======================= FSM types =======================
// Defined before any function: the Arduino builder inserts auto-generated
// function prototypes above the first function, and they need these types.

enum State : uint8_t { STATE_NORMAL, STATE_HALT };

enum FaultType : uint8_t {
  FAULT_OVER_TEMP,
  FAULT_OVER_CURRENT,
  FAULT_NO_REPLY,
  FAULT_SETUP_MISMATCH,
  FAULT_BAD_INPUT
};

struct Fault {
  FaultType type;
  uint8_t   id;      // servo ID (unused for FAULT_BAD_INPUT)
  long      value;   // degC, mA, consecutive missed reads, or setup attempts
};

// ======================= STS3215 protocol =======================

const uint8_t REG_TORQUE_ENABLE       = 0x28;  // 40
const uint8_t REG_GOAL_POSITION       = 0x2A;  // 42, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_POSITION    = 0x38;  // 56, 2 bytes, sign-magnitude
const uint8_t REG_PRESENT_TEMPERATURE = 0x3F;  // 63, 1 byte, degrees C
const uint8_t REG_PRESENT_CURRENT     = 0x45;  // 69, 2 bytes, sign-magnitude

// One SYNC READ covers Present_Position .. Present_Current (registers 56..70)
const uint8_t READ_START = REG_PRESENT_POSITION;
const uint8_t READ_LEN   = REG_PRESENT_CURRENT + 2 - READ_START;   // 15 bytes

// Offsets of each field inside the block
const uint8_t OFS_POSITION    = REG_PRESENT_POSITION    - READ_START;
const uint8_t OFS_TEMPERATURE = REG_PRESENT_TEMPERATURE - READ_START;
const uint8_t OFS_CURRENT     = REG_PRESENT_CURRENT     - READ_START;

const uint8_t INST_READ       = 0x02;
const uint8_t INST_WRITE      = 0x03;
const uint8_t INST_SYNC_READ  = 0x82;
const uint8_t INST_SYNC_WRITE = 0x83;
const uint8_t BROADCAST_ID    = 0xFE;

uint8_t lastTx[64];                      // largest packet is the SYNC WRITE (26 bytes)
uint8_t lastTxLen = 0;

// Build and send a packet; remember it so we can ignore its echo
void sendPacket(uint8_t id, uint8_t instr, const uint8_t* params, uint8_t n) {
  uint8_t len = n + 2;
  uint8_t sum = id + len + instr;
  lastTxLen = 0;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = 0xFF;
  lastTx[lastTxLen++] = id;
  lastTx[lastTxLen++] = len;
  lastTx[lastTxLen++] = instr;
  for (uint8_t i = 0; i < n; i++) {
    lastTx[lastTxLen++] = params[i];
    sum += params[i];
  }
  lastTx[lastTxLen++] = (uint8_t)~sum;

  while (Serial1.available()) Serial1.read();  // clear stale bytes
  Serial1.write(lastTx, lastTxLen);
  Serial1.flush();
}

// Read one byte with timeout; returns -1 on timeout
int readByte(unsigned long deadline) {
  while ((long)(deadline - micros()) > 0) {
    if (Serial1.available()) return Serial1.read();
  }
  return -1;
}

// Wait for a valid status packet from `id`, skipping our own echo.
// Fills data[] with the payload; returns payload length or -1.
int readStatus(uint8_t id, uint8_t* data, uint8_t maxData) {
  unsigned long deadline = micros() + REPLY_TIMEOUT_US;

  while ((long)(deadline - micros()) > 0) {
    // Find header 0xFF 0xFF
    int b = readByte(deadline);
    if (b != 0xFF) continue;
    b = readByte(deadline);
    if (b != 0xFF) continue;

    int rid = readByte(deadline);
    if (rid == 0xFF) rid = readByte(deadline);   // tolerate extra 0xFF
    int len = readByte(deadline);
    if (rid < 0 || len < 2 || len > maxData + 2) continue;

    uint8_t pkt[4 + READ_LEN + 2];               // header + id + len + err + data + checksum
    if (4 + len > (int)sizeof(pkt)) continue;
    uint8_t p = 0;
    pkt[p++] = 0xFF; pkt[p++] = 0xFF; pkt[p++] = rid; pkt[p++] = len;

    bool ok = true;
    for (int i = 0; i < len; i++) {
      int c = readByte(deadline);
      if (c < 0) { ok = false; break; }
      pkt[p++] = (uint8_t)c;
    }
    if (!ok) return -1;

    // Skip the echo of the packet we just sent
    if (p == lastTxLen && memcmp(pkt, lastTx, p) == 0) continue;

    // Verify checksum
    uint8_t sum = rid + len;
    for (int i = 4; i < p - 1; i++) sum += pkt[i];
    if ((uint8_t)~sum != pkt[p - 1]) continue;
    if (rid != id) continue;

    // pkt[4] = error byte, payload follows
    uint8_t n = len - 2;
    memcpy(data, &pkt[5], n);
    return n;
  }
  return -1;
}

void writeByte(uint8_t id, uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(id, INST_WRITE, p, 2);
  uint8_t dummy[4];
  readStatus(id, dummy, sizeof(dummy));  // consume the servo's ack
}

// Writes one byte to every servo at once. Broadcast packets get no reply.
void broadcastWriteByte(uint8_t addr, uint8_t val) {
  uint8_t p[] = { addr, val };
  sendPacket(BROADCAST_ID, INST_WRITE, p, 2);
}

// STS 16-bit values are little-endian with bit 15 as the sign
int decodeSignMag(uint8_t lo, uint8_t hi) {
  uint16_t raw = lo | (hi << 8);
  return (raw & 0x8000) ? -(int)(raw & 0x7FFF) : (int)raw;
}

void encodeSignMag(int v, uint8_t& lo, uint8_t& hi) {
  uint16_t raw = (v < 0) ? (uint16_t)(0x8000 | ((-v) & 0x7FFF)) : (uint16_t)(v & 0x7FFF);
  lo = raw & 0xFF;
  hi = raw >> 8;
}

// Reads a 2-byte register from one servo; returns false if it didn't reply
bool readWord(uint8_t id, uint8_t addr, int& val) {
  uint8_t p[] = { addr, 2 };
  sendPacket(id, INST_READ, p, 2);
  uint8_t data[2];
  if (readStatus(id, data, sizeof(data)) != 2) return false;
  val = decodeSignMag(data[0], data[1]);
  return true;
}

// Sends Goal_Position to all servos with one SYNC WRITE (no reply)
void writeGoalPositions(const int* goal) {
  uint8_t p[2 + 3 * NUM_SERVOS];
  uint8_t k = 0;
  p[k++] = REG_GOAL_POSITION;
  p[k++] = 2;                                // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    p[k++] = IDS[i];
    encodeSignMag(goal[i], p[k], p[k + 1]);
    k += 2;
  }
  sendPacket(BROADCAST_ID, INST_SYNC_WRITE, p, k);
}

// Reads position, temperature and current of all servos with one SYNC READ.
// replied[i] tells whether servo i answered; its other fields are valid only if so.
// Returns true if all replied.
bool readAllServos(int* ticks, int* tempC, int* currRaw, bool* replied) {
  uint8_t p[2 + NUM_SERVOS];
  p[0] = READ_START;
  p[1] = READ_LEN;                           // bytes per servo
  for (uint8_t i = 0; i < NUM_SERVOS; i++) p[2 + i] = IDS[i];
  sendPacket(BROADCAST_ID, INST_SYNC_READ, p, sizeof(p));

  // Servos reply one after another, in the order of the ID list
  bool ok = true;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    uint8_t data[READ_LEN];
    int n = readStatus(IDS[i], data, sizeof(data));
    replied[i] = (n == READ_LEN);
    if (!replied[i]) {
      ok = false;
      continue;
    }
    ticks[i]   = decodeSignMag(data[OFS_POSITION], data[OFS_POSITION + 1]);
    tempC[i]   = data[OFS_TEMPERATURE];
    currRaw[i] = decodeSignMag(data[OFS_CURRENT], data[OFS_CURRENT + 1]);
  }
  return ok;
}

// ======================= Shared data =======================

State state = STATE_NORMAL;

// Latest sensor data (valid for servo i only if replied[i] is true)
int  ticks[NUM_SERVOS];
int  tempC[NUM_SERVOS];
int  currRaw[NUM_SERVOS];
bool replied[NUM_SERVOS];

// Goal position of each servo, always inside [MIN_TICKS, MAX_TICKS]
// except at startup, where it equals the present position
int targetTicks[NUM_SERVOS];

// Consecutive missed readings per servo
uint8_t missedReads[NUM_SERVOS] = {0};

// A servo can trip at most two conditions at once (temp + current, or no reply)
const uint8_t MAX_FAULTS = 2 * NUM_SERVOS;
Fault   faults[MAX_FAULTS];
uint8_t numFaults = 0;

// The rejected terminal line and why it was rejected (for the HALT message)
char        badInputText[INPUT_LINE_LEN] = "";
const char* badInputWhy = "";

// Latest kinematics. The last good FK result is kept so a dropped reply
// doesn't blank the output.
float    q[NUM_FK_JOINTS] = {0};
float    eePos[3] = {0, 0, 0};                     // end-effector position [m]
float    eeRot[9] = {1, 0, 0,  0, 1, 0,  0, 0, 1}; // end-effector rotation, column-major
bool     haveFK   = false;   // at least one FK result available
bool     fkFresh  = false;   // FK succeeded on the most recent read
uint32_t fkMicros = 0;       // time taken by the last FK evaluation

// ======================= Angle helpers =======================

// Converts a servo's ticks to radians relative to its zero point
float ticksToAngle(uint8_t i, int t) {
  return (t - ZERO_TICKS[i]) * RAD_PER_TICK;
}

// Sets the target of servo i from an angle in radians, clamped to the joint limits
void setTarget(uint8_t i, float rad) {
  float t = ZERO_TICKS[i] + rad / RAD_PER_TICK;
  bool clamped = false;
  if (t < MIN_TICKS[i]) { t = MIN_TICKS[i]; clamped = true; }
  if (t > MAX_TICKS[i]) { t = MAX_TICKS[i]; clamped = true; }
  targetTicks[i] = (int)lroundf(t);

  Serial.print("Target servo ");
  Serial.print(IDS[i]);
  Serial.print(": ");
  Serial.print(ticksToAngle(i, targetTicks[i]), 3);
  Serial.print(" rad (");
  Serial.print(targetTicks[i]);
  Serial.print(" ticks)");
  if (clamped) {
    Serial.print(" - requested ");
    Serial.print(rad, 3);
    Serial.print(" is outside [");
    Serial.print(ticksToAngle(i, MIN_TICKS[i]), 3);
    Serial.print(", ");
    Serial.print(ticksToAngle(i, MAX_TICKS[i]), 3);
    Serial.print("], clamped");
  }
  Serial.println();
}

// ======================= Forward kinematics =======================

// Wraps the CasADi function so101_fk:(q[5]) -> (p[3], R[3x3]).
// p is in meters, in the robot base frame.
// R is stored column-major (CasADi convention): R(row, col) = R[row + 3*col].
bool computeFK(const float* q, float* p, float* R) {
  const casadi_real* arg[so101_fk_SZ_ARG] = { q };
  casadi_real*       res[so101_fk_SZ_RES] = { p, R };
  // SZ_IW and SZ_W are 0, so no work buffers are needed
  return so101_fk(arg, res, NULL, NULL, 0) == 0;
}

inline float Rat(const float* R, int row, int col) { return R[row + 3 * col]; }

// ZYX Euler angles (roll about x, pitch about y, yaw about z), in radians
void rotToRPY(const float* R, float& roll, float& pitch, float& yaw) {
  yaw   = atan2f(Rat(R, 1, 0), Rat(R, 0, 0));
  pitch = atan2f(-Rat(R, 2, 0), sqrtf(Rat(R, 2, 1) * Rat(R, 2, 1) + Rat(R, 2, 2) * Rat(R, 2, 2)));
  roll  = atan2f(Rat(R, 2, 1), Rat(R, 2, 2));
}

// Converts the latest reading to joint angles and runs FK.
// FK runs only when all arm joints replied; otherwise the last result is kept.
void updateKinematics() {
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    if (!replied[i]) { fkFresh = false; return; }
  }
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    q[i] = JOINT_SIGN[i] * ticksToAngle(i, ticks[i]) + JOINT_OFFSET[i];
  }
  uint32_t t0 = micros();
  fkFresh  = computeFK(q, eePos, eeRot);
  fkMicros = micros() - t0;
  haveFK   = haveFK || fkFresh;
}

// ======================= Terminal input =======================

void printUsage() {
  Serial.print("Usage: \"<a1> ... <a");
  Serial.print(NUM_SERVOS);
  Serial.println(">\", angles in rad. Any other input halts the robot.");
}

// Records the rejected line and goes to HALT
void haltOnBadInput(const char* text, const char* why) {
  strncpy(badInputText, text, INPUT_LINE_LEN - 1);
  badInputText[INPUT_LINE_LEN - 1] = '\0';
  badInputWhy = why;

  numFaults = 0;
  addFault(FAULT_BAD_INPUT, 0, 0);
  transitionTo(STATE_HALT);
}

void handleCommand(char* line) {
  if (state != STATE_NORMAL) {
    Serial.println("Command ignored: robot is in HALT");
    return;
  }

  // Keep an untouched copy for the HALT message (strtok() modifies line)
  char original[INPUT_LINE_LEN];
  strncpy(original, line, INPUT_LINE_LEN - 1);
  original[INPUT_LINE_LEN - 1] = '\0';

  // Parse all numbers first, so a bad line changes no target
  float v[NUM_SERVOS];
  uint8_t n = 0;
  for (char* tok = strtok(line, " ,;\t"); tok != NULL; tok = strtok(NULL, " ,;\t")) {
    if (n == NUM_SERVOS) {
      haltOnBadInput(original, "too many values");
      return;
    }
    char* end;
    v[n] = strtod(tok, &end);
    if (end == tok || *end != '\0' || !isfinite(v[n])) {
      haltOnBadInput(original, "not a number");
      return;
    }
    n++;
  }

  if (n != NUM_SERVOS) {
    haltOnBadInput(original, "too few values");
    return;
  }

  for (uint8_t i = 0; i < NUM_SERVOS; i++) setTarget(i, v[i]);
}

// Collects characters into a line without blocking; runs a command on Enter
void pollTerminal() {
  static char    line[INPUT_LINE_LEN];
  static uint8_t len = 0;
  static bool    overflow = false;

  while (Serial.available()) {
    char c = Serial.read();
    if (c == '\n' || c == '\r') {
      if (overflow) {
        line[len] = '\0';                       // first INPUT_LINE_LEN-1 characters
        if (state == STATE_NORMAL) {
          haltOnBadInput(line, "line too long");
        } else {
          Serial.println("Command ignored: robot is in HALT");
        }
      } else if (len > 0) {
        line[len] = '\0';
        handleCommand(line);
      }
      // Empty lines are ignored: "\r\n" line endings produce one after every command
      len = 0;
      overflow = false;
    } else if (len < INPUT_LINE_LEN - 1) {
      line[len++] = c;
    } else {
      overflow = true;
    }
  }
}

// ======================= Finite State Machine =======================

const char* stateName(State s) {
  switch (s) {
    case STATE_NORMAL: return "NORMAL";
    case STATE_HALT:   return "HALT";
  }
  return "UNKNOWN";
}

void addFault(FaultType type, uint8_t id, long value) {
  if (numFaults < MAX_FAULTS) faults[numFaults++] = { type, id, value };
}

// Guard of the NORMAL -> HALT transition, evaluated on each new reading.
// Records every condition that is currently true; returns true if any.
bool checkHaltConditions() {
  numFaults = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) {
      if (missedReads[i] < 255) missedReads[i]++;
      if (missedReads[i] > MAX_MISSED_READS) {
        addFault(FAULT_NO_REPLY, IDS[i], missedReads[i]);
      }
      continue;   // temperature and current are not valid for this servo
    }
    missedReads[i] = 0;

    if (tempC[i] > MAX_TEMP_C) {
      addFault(FAULT_OVER_TEMP, IDS[i], tempC[i]);
    }

    long mA = labs(lroundf(currRaw[i] * MA_PER_CURRENT_UNIT));
    if (mA > MAX_CURRENT_MA) {
      addFault(FAULT_OVER_CURRENT, IDS[i], mA);
    }
  }
  return numFaults > 0;
}

void printFault(const Fault& f) {
  if (f.type == FAULT_BAD_INPUT) {
    Serial.print("  reason: input \"");
    Serial.print(badInputText);
    Serial.print("\" rejected (");
    Serial.print(badInputWhy);
    Serial.print("), expected ");
    Serial.print(NUM_SERVOS);
    Serial.println(" angles");
    return;
  }

  Serial.print("  reason: servo ");
  Serial.print(f.id);
  switch (f.type) {
    case FAULT_OVER_TEMP:
      Serial.print(" temperature ");
      Serial.print(f.value);
      Serial.print(" C > ");
      Serial.print(MAX_TEMP_C);
      Serial.println(" C");
      break;
    case FAULT_OVER_CURRENT:
      Serial.print(" current ");
      Serial.print(f.value);
      Serial.print(" mA > ");
      Serial.print(MAX_CURRENT_MA);
      Serial.println(" mA");
      break;
    case FAULT_NO_REPLY:
      Serial.print(" not responding for ");
      Serial.print(f.value);
      Serial.print(" consecutive readings > ");
      Serial.println(MAX_MISSED_READS);
      break;
    case FAULT_SETUP_MISMATCH:
      Serial.print(" goal position could not be aligned with present position after ");
      Serial.print(f.value);
      Serial.println(" attempts");
      break;
    case FAULT_BAD_INPUT:
      break;   // handled above
  }
}

void transitionTo(State next) {
  // Entry action of HALT first, before the (slow) printing
  if (next == STATE_HALT) broadcastWriteByte(REG_TORQUE_ENABLE, 0);

  Serial.print("TRANSITION ");
  Serial.print(stateName(state));
  Serial.print(" -> ");
  Serial.println(stateName(next));
  if (next == STATE_HALT) {
    for (uint8_t k = 0; k < numFaults; k++) printFault(faults[k]);
  }
  state = next;
}

// ----- State behaviours -----

// NORMAL: at 30 Hz send the targets, read the servos, update FK, evaluate the guard
void runNormalState() {
  static unsigned long nextCycle = micros();
  if ((long)(micros() - nextCycle) < 0) return;

  // The deadline advances by exactly one period, so the rate doesn't drift.
  nextCycle += CYCLE_PERIOD_US;
  // If we fell more than a full period behind, resynchronize
  if ((long)(micros() - nextCycle) >= 0) nextCycle = micros() + CYCLE_PERIOD_US;

  writeGoalPositions(targetTicks);
  readAllServos(ticks, tempC, currRaw, replied);
  updateKinematics();

  if (checkHaltConditions()) transitionTo(STATE_HALT);
}

// HALT: keep torque disabled on every servo (re-sent periodically, so a servo
// that missed the first packet or rebooted is still switched off)
void runHaltState() {
  static unsigned long lastSent = millis();
  if (millis() - lastSent >= HALT_REPEAT_MS) {
    lastSent = millis();
    broadcastWriteByte(REG_TORQUE_ENABLE, 0);
  }
}

// ======================= Printing =======================

// Prints a value with a leading space for positives so columns line up
void printAligned(float v, uint8_t decimals) {
  if (v >= 0) Serial.print(' ');
  Serial.print(v, decimals);
}

// Prints an integer right-aligned in a field of `width` characters
void printPadded(long v, uint8_t width) {
  char buf[16];
  snprintf(buf, sizeof(buf), "%*ld", width, v);
  Serial.print(buf);
}

// Prints one status line: state, joints/FK, temperature/current
void printStatus() {
  // State, padded so the columns stay put when it changes
  char st[16];
  snprintf(st, sizeof(st), "%-6s", stateName(state));
  Serial.print("State: ");
  Serial.print(st);

  // Joints, gripper, FK
  Serial.print(" | q[rad] ");
  for (uint8_t i = 0; i < NUM_FK_JOINTS; i++) {
    printAligned(q[i], 3);
    Serial.print(' ');
  }

  const uint8_t g = NUM_SERVOS - 1;   // gripper index
  Serial.print("| grip ");
  if (!replied[g]) Serial.print("  N/A ");
  else printAligned(ticksToAngle(g, ticks[g]), 3);

  if (haveFK) {
    float roll, pitch, yaw;
    rotToRPY(eeRot, roll, pitch, yaw);

    Serial.print(" | xyz[mm] ");
    for (uint8_t k = 0; k < 3; k++) {
      printAligned(eePos[k] * 1000.0f, 1);
      Serial.print(' ');
    }
    Serial.print("| rpy[deg] ");
    printAligned(roll  * RAD_TO_DEG, 1); Serial.print(' ');
    printAligned(pitch * RAD_TO_DEG, 1); Serial.print(' ');
    printAligned(yaw   * RAD_TO_DEG, 1);
    Serial.print(" | fk ");
    Serial.print(fkMicros);
    Serial.print("us");
    if (!fkFresh) Serial.print(" (stale)");
  } else {
    Serial.print(" | FK: waiting for all joints");
  }

  // Temperature and current per servo (order of IDS)
  Serial.print(" | T[C] ");
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) Serial.print(" N/A");
    else printPadded(tempC[i], 4);
  }
  Serial.print(" | I[mA] ");
  long totalmA = 0;
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (!replied[i]) { Serial.print("   N/A"); continue; }
    long mA = lroundf(currRaw[i] * MA_PER_CURRENT_UNIT);
    totalmA += labs(mA);
    printPadded(mA, 6);
  }
  Serial.print(" | total ");
  Serial.print(totalmA);
  Serial.println("mA");
}

// ======================= Setup =======================

// Reads Goal_Position back from every servo; returns the index of the first
// servo whose goal differs from targetTicks, or -1 if all match.
int findGoalMismatch() {
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    int goal;
    if (!readWord(IDS[i], REG_GOAL_POSITION, goal) || goal != targetTicks[i]) return i;
  }
  return -1;
}

void setup() {
  Serial.begin(115200);
  Serial1.begin(BAUD);
  delay(200);

  // Sanity check that the generated function matches what this sketch expects
  const casadi_int* sp = so101_fk_sparsity_in(0);
  if (so101_fk_n_in() != 1 || so101_fk_n_out() != 2 || sp[0] != NUM_FK_JOINTS) {
    Serial.println("ERROR: so101_fk signature does not match NUM_FK_JOINTS");
  }

  Serial.println("Setup: aligning goal positions with present positions");

  // Before enabling torque, make every goal equal to the present position,
  // so the arm doesn't jump when the servos start holding.
  int mismatch = -1;
  bool aligned = false;
  for (uint8_t attempt = 1; attempt <= MAX_SETUP_ATTEMPTS && !aligned; attempt++) {
    if (attempt > 1) delay(CYCLE_PERIOD_US / 1000);

    bool allReplied = readAllServos(ticks, tempC, currRaw, replied);
    updateKinematics();
    if (checkHaltConditions()) {
      transitionTo(STATE_HALT);
      return;
    }
    if (!allReplied) continue;

    for (uint8_t i = 0; i < NUM_SERVOS; i++) targetTicks[i] = ticks[i];
    writeGoalPositions(targetTicks);

    mismatch = findGoalMismatch();
    aligned  = (mismatch < 0);
  }

  if (!aligned) {
    numFaults = 0;
    addFault(FAULT_SETUP_MISMATCH, mismatch >= 0 ? IDS[mismatch] : 0, MAX_SETUP_ATTEMPTS);
    transitionTo(STATE_HALT);
    return;
  }

  // Goals verified: now it's safe to enable torque
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    writeByte(IDS[i], REG_TORQUE_ENABLE, 1);
  }

  Serial.print("Setup: torque enabled, start angles [rad]");
  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    Serial.print(' ');
    Serial.print(ticksToAngle(i, targetTicks[i]), 3);
  }
  Serial.println();

  for (uint8_t i = 0; i < NUM_SERVOS; i++) {
    if (targetTicks[i] < MIN_TICKS[i] || targetTicks[i] > MAX_TICKS[i]) {
      Serial.print("Warning: servo ");
      Serial.print(IDS[i]);
      Serial.println(" starts outside its limits; the next command will bring it inside");
    }
  }
  printUsage();
}

// ======================= Main =======================

void loop() {
  static unsigned long lastPrint = 0;

  pollTerminal();

  switch (state) {
    case STATE_NORMAL: runNormalState(); break;
    case STATE_HALT:   runHaltState();   break;
  }

  // ---- Print the status line at 5 Hz ----
  if (millis() - lastPrint >= PRINT_PERIOD_MS) {
    lastPrint = millis();
    printStatus();
  }
}
```

# Think About It

1. Our halt state removes the torque, and the arm falls. Can you think of situations where this is the wrong reaction? What could a real robot do instead (hold its position, perform a controlled descent, engage mechanical brakes)?
2. Were you able to trip the governor by pushing the arm in step 3 of the test sequence? What does this tell you about using 2700 mA as a threshold? What value would you choose, and how would you choose it?
3. HALT is latching: a fault that clears by itself doesn't bring the robot back to normal operation. Why is this a good idea? What would a safe procedure to resume operation look like?
4. What happens if the MCU crashes or is disconnected while the robot is in the normal state? Which safety layer is still active then?
5. The limits of Deliverable 2.3 are the joints' mechanical end stops. What happens if you command a joint exactly to its limit and keep it there?

# Optional Extension: Smoother and Safer Motion

With the reference solution of Deliverable 2.4, a new target is sent straight to the servos, which move there as fast as their controllers allow. Some ideas to improve on this:

- **Ramp the targets in software:** instead of jumping to the requested target, move `targetTicks` towards it a little at every 30 Hz cycle, limiting the change per cycle. This bounds the speed of every joint.
- **Limit speed and acceleration in the servo:** the ST3215 has acceleration and goal speed registers (look them up in the [register table](https://pypi.org/project/st3215/)). They live in SRAM, so writing them does not wear the EEPROM.
- **Add a safety margin to the joint limits:** shrink each range measured in Deliverable 2.3 by a few dozen ticks, so the joints are never commanded into their mechanical end stops.